In [ ]:
import sys, os, warnings, re, calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
from pathlib import Path
warnings.filterwarnings("ignore")

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / "src" / "agsi_client").exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"Project root: {_c}"); break

# ═══════════════════════════════════════════════
# CONFIGURATION — edit these lines to change targets
TARGET  = "Mar 2027"     # single-target analysis (cells 1–4)
TARGETS = ["Mar 2027", "Summer 26", "Winter 26", "CAL 2026"]  # multi-target overlay (cell 5)

# Supported TARGET formats:
#   "Mar 2027"   — single month (month name + 4-digit year)
#   "Q3 2027"    — quarter (Q1–Q4 + 4-digit year; Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec)
#   "Summer 25"  — Apr–Sep (2 or 4 digit year)
#   "Winter 25"  — Oct 25 – Mar 26 (year is the Oct start year)
#   "CAL 2026"   — full calendar year Jan–Dec
# ═══════════════════════════════════════════════

RAW = Path("data/raw")
ttf = pd.read_csv(RAW / "ttf_curve.csv", index_col=0, parse_dates=True)
ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
ttf = ttf.sort_index()

print(f"TTF curve loaded: {ttf.shape}")
print(f"  {ttf.index[0].date()} -> {ttf.index[-1].date()}")
print(f"  Columns: {list(ttf.columns[:6])} ...")
print(f"\nSingle target : {TARGET}")
print(f"Multi targets : {TARGETS}")

In [ ]:
# ── Month name lookup (abbr + full, case-insensitive) ───────────────────────────────────────────
_MONTH_MAP = {}
for _i in range(1, 13):
    _MONTH_MAP[calendar.month_abbr[_i].lower()] = _i
    _MONTH_MAP[calendar.month_name[_i].lower()]  = _i


def parse_target(target_str):
    """
    Return list of (delivery_year, delivery_month) tuples for a contract string.

    Formats:
      "Mar 2027"    -> [(2027, 3)]
      "March 2027"  -> [(2027, 3)]
      "Q3 2027"     -> [(2027, 7), (2027, 8), (2027, 9)]
      "Summer 25"   -> [(2025, 4), ..., (2025, 9)]   Apr-Sep
      "Winter 25"   -> [(2025,10), ..., (2026, 3)]   Oct-Mar
      "CAL 2026"    -> [(2026, 1), ..., (2026,12)]
    """
    s = target_str.strip()

    # CAL YYYY
    m = re.match(r"^CAL\s+(\d{4})$", s, re.I)
    if m:
        yr = int(m.group(1))
        return [(yr, mo) for mo in range(1, 13)]

    # Q[1-4] YYYY
    m = re.match(r"^Q([1-4])\s+(\d{4})$", s, re.I)
    if m:
        q, yr = int(m.group(1)), int(m.group(2))
        start = (q - 1) * 3 + 1
        return [(yr, mo) for mo in range(start, start + 3)]

    # Summer YY or Summer YYYY
    m = re.match(r"^Summer\s+(\d{2,4})$", s, re.I)
    if m:
        raw = m.group(1)
        yr = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, mo) for mo in range(4, 10)]

    # Winter YY or Winter YYYY  ->  Oct of yr through Mar of yr+1
    m = re.match(r"^Winter\s+(\d{2,4})$", s, re.I)
    if m:
        raw = m.group(1)
        yr = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, 10), (yr, 11), (yr, 12), (yr+1, 1), (yr+1, 2), (yr+1, 3)]

    # Month YYYY  (abbr or full)
    m = re.match(r"^(\w+)\s+(\d{4})$", s)
    if m:
        mo_str, yr = m.group(1).lower(), int(m.group(2))
        if mo_str in _MONTH_MAP:
            return [(yr, _MONTH_MAP[mo_str])]

    raise ValueError(
        f"Cannot parse: {target_str!r}\n"
        "Supported: 'Mar 2027', 'Q3 2027', 'Summer 25', 'Winter 25', 'CAL 2026'"
    )


# ── Test all 5 formats ─────────────────────────────────────────────────────────────────────
_test_cases = [
    "Mar 2027",
    "Q3 2027",
    "Summer 25",
    "Winter 25",
    "CAL 2026",
]
for _tc in _test_cases:
    _dm = parse_target(_tc)
    _span = f"{calendar.month_abbr[_dm[0][1]]} {_dm[0][0]}"
    if len(_dm) > 1:
        _span += f" - {calendar.month_abbr[_dm[-1][1]]} {_dm[-1][0]}"
    print(f"  {_tc:<18} -> {len(_dm):>2} month(s): {_span}")

print()
delivery_months = parse_target(TARGET)
print(f"TARGET: {TARGET!r}")
print(f"Delivery months ({len(delivery_months)} total):")
for yr, mo in delivery_months:
    print(f"  {calendar.month_abbr[mo]} {yr}")

In [ ]:
def build_price_series(ttf_df, delivery_months):
    """
    Return pd.Series (trade_date -> avg_price) for the target delivery period.

    A date is included ONLY if ALL delivery months are simultaneously quotable:
      1 <= months_ahead <= 24  for every (del_yr, del_mo)
    AND all M+N values are non-NaN.

    months_ahead = (del_yr - trade_dt.year)*12 + (del_mo - trade_dt.month)

    ICE Endex TTF futures expire ~2 business days before delivery month starts,
    so a contract is no longer quotable once months_ahead < 1 (delivery has begun
    or the contract has rolled off).  The M24 cap reflects the maximum tenor
    available in the curve.
    """
    rows = []
    for trade_dt, trade_row in ttf_df.iterrows():
        prices = []
        all_valid = True
        for (del_yr, del_mo) in delivery_months:
            months_ahead = (del_yr - trade_dt.year) * 12 + (del_mo - trade_dt.month)
            if not (1 <= months_ahead <= 24):
                all_valid = False
                break
            col = f"M{months_ahead}"
            if col not in trade_row.index or pd.isna(trade_row[col]):
                all_valid = False
                break
            prices.append(float(trade_row[col]))
        if all_valid and prices:
            rows.append({"date": trade_dt, "price": np.mean(prices)})
    if not rows:
        return pd.Series(dtype=float, name="price")
    df_out = pd.DataFrame(rows).set_index("date").sort_index()
    return df_out["price"]


price_series = build_price_series(ttf, delivery_months)

if price_series.empty:
    print(f"No quotable dates found for {TARGET}.")
    print(f"  Delivery months span: {delivery_months[0]} -> {delivery_months[-1]}")
    print(f"  TTF data range: {ttf.index[0].date()} -> {ttf.index[-1].date()}")
    print()
    print("Reason: for a trade date to be included, ALL delivery legs must satisfy")
    print("  1 <= months_ahead <= 24  simultaneously. If any leg has already expired")
    print("  (months_ahead < 1) or is too far out (months_ahead > 24), the trade")
    print("  date is excluded entirely.")
else:
    first_dt  = price_series.index[0].date()
    last_dt   = price_series.index[-1].date()
    first_px  = float(price_series.iloc[0])
    last_px   = float(price_series.iloc[-1])
    n_obs     = len(price_series)

    # Human-readable delivery span
    del_start = f"{calendar.month_abbr[delivery_months[0][1]]} {delivery_months[0][0]}"
    del_end   = f"{calendar.month_abbr[delivery_months[-1][1]]} {delivery_months[-1][0]}"
    if len(delivery_months) == 1:
        del_desc = del_start
    else:
        del_desc = f"{del_start} - {del_end}"

    print(f"Price series for {TARGET!r}")
    print(f"  First quotable date : {first_dt}  ->  {first_px:.3f} EUR/MWh")
    print(f"  Last quotable date  : {last_dt}  ->  {last_px:.3f} EUR/MWh")
    print(f"  N observations      : {n_obs:,} trading days")
    print()
    print(f"  Explanation: {TARGET} (delivery {del_desc}) is quotable from")
    print(f"    {first_dt} — the first date when all {len(delivery_months)} delivery month(s)")
    print(f"    are within M1-M24 simultaneously")
    print(f"    to {last_dt} — the last date before the earliest delivery leg")
    print(f"    ({del_start}) rolls to months_ahead < 1 (ICE TTF expiry)")

In [ ]:
if price_series.empty:
    print(f"No data to chart for {TARGET}.")
else:
    ps = price_series

    first_dt = ps.index[0]
    last_dt  = ps.index[-1]
    idx_min  = ps.idxmin()
    idx_max  = ps.idxmax()
    p_first  = float(ps.iloc[0])
    p_last   = float(ps.iloc[-1])
    p_min    = float(ps.min())
    p_max    = float(ps.max())

    pad = (p_max - p_min) * 0.05
    y_range = [p_min - pad, p_max + pad]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=ps.index, y=ps.values,
        mode="lines",
        line=dict(color="#2E75B6", width=1.8),
        name=TARGET,
    ))

    # Deduplicate annotation positions so overlapping points don't double-print
    ann_raw = [
        dict(x=first_dt, y=p_first, text=f"First<br>{p_first:.2f}",
             showarrow=True, arrowhead=2, ax=40, ay=-30, font=dict(color="#555")),
        dict(x=last_dt,  y=p_last,  text=f"Latest<br>{p_last:.2f}",
             showarrow=True, arrowhead=2, ax=-45, ay=-30, font=dict(color="#E74C3C")),
        dict(x=idx_min,  y=p_min,   text=f"Min {p_min:.2f}<br>{idx_min.strftime('%b %Y')}",
             showarrow=True, arrowhead=2, ax=0, ay=40,  font=dict(color="#27AE60")),
        dict(x=idx_max,  y=p_max,   text=f"Max {p_max:.2f}<br>{idx_max.strftime('%b %Y')}",
             showarrow=True, arrowhead=2, ax=0, ay=-40, font=dict(color="#E74C3C")),
    ]
    seen_pts = set()
    annotations = []
    for ann in ann_raw:
        key = (ann["x"], round(ann["y"], 1))
        if key not in seen_pts:
            seen_pts.add(key)
            annotations.append(ann)

    title_window = f"{first_dt.date()} to {last_dt.date()}"

    fig.update_layout(
        title=f"Market-Implied Price for {TARGET} | Quotable window: {title_window}",
        xaxis_title="Trade Date",
        yaxis=dict(title="Price (EUR/MWh)", range=y_range),
        annotations=annotations,
        height=480,
        template="plotly_white",
    )
    fig.show()

In [ ]:
if price_series.empty:
    print(f"No data for {TARGET}.")
else:
    ps = price_series.dropna()

    # Realized annualized volatility (log returns)
    log_ret = np.log(ps / ps.shift(1)).dropna()
    ann_vol = float(log_ret.std() * np.sqrt(252)) * 100   # %

    # Max drawdown (from rolling peak)
    roll_max = ps.cummax()
    drawdown = (ps - roll_max) / roll_max * 100
    max_dd   = float(drawdown.min())

    # % change first to last
    pct_chg = (ps.iloc[-1] / ps.iloc[0] - 1) * 100

    del_start = f"{calendar.month_abbr[delivery_months[0][1]]} {delivery_months[0][0]}"
    del_end   = f"{calendar.month_abbr[delivery_months[-1][1]]} {delivery_months[-1][0]}"
    del_period = del_start if len(delivery_months) == 1 else f"{del_start} - {del_end}"

    print(f"Contract Price Statistics -- {TARGET}")
    print("=" * 56)
    print(f"  Target           : {TARGET}")
    print(f"  Delivery period  : {del_period} ({len(delivery_months)} month(s))")
    print(f"  Quotable window  : {ps.index[0].date()} -> {ps.index[-1].date()}")
    print(f"  N trading days   : {len(ps):,}")
    print(f"  First price      : {ps.iloc[0]:.3f} EUR/MWh")
    print(f"  Last price       : {ps.iloc[-1]:.3f} EUR/MWh")
    print(f"  % change         : {pct_chg:+.1f}%")
    print(f"  Min              : {ps.min():.3f}  ({ps.idxmin().date()})")
    print(f"  Max              : {ps.max():.3f}  ({ps.idxmax().date()})")
    print(f"  Realized vol     : {ann_vol:.1f}% p.a. (annualized, log-return)")
    print(f"  Max drawdown     : {max_dd:.1f}%")

In [ ]:
COLORS_LIST = pc.qualitative.Set1 + pc.qualitative.Set2

fig_abs  = go.Figure()   # absolute prices
fig_norm = go.Figure()   # normalized to % change from first observation

print(f"Building price history for {len(TARGETS)} contracts...")
for i, tgt in enumerate(TARGETS):
    try:
        dm = parse_target(tgt)
    except ValueError as e:
        print(f"  Skipping {tgt!r}: {e}")
        continue

    ps = build_price_series(ttf, dm)
    if ps.empty:
        print(f"  Warning: {tgt} has no quotable dates in TTF data range -- skipping")
        continue

    ps = ps.dropna()
    color   = COLORS_LIST[i % len(COLORS_LIST)]
    p_first = float(ps.iloc[0])
    ps_norm = (ps / p_first - 1) * 100   # % change from first observation

    window_str = f"{ps.index[0].date()} to {ps.index[-1].date()}"
    legend_name = f"{tgt} ({window_str})"

    print(f"  {tgt:<18}: {len(ps):>4} obs  "
          f"{ps.index[0].date()} -> {ps.index[-1].date()}  "
          f"first={p_first:.2f}  latest={ps.iloc[-1]:.2f}")

    fig_abs.add_trace(go.Scatter(
        x=ps.index, y=ps.values,
        name=legend_name, mode="lines",
        line=dict(color=color, width=1.8),
    ))
    fig_norm.add_trace(go.Scatter(
        x=ps_norm.index, y=ps_norm.values,
        name=legend_name, mode="lines",
        line=dict(color=color, width=1.8),
    ))

fig_abs.update_layout(
    title="Contract Price Evolution -- Absolute (EUR/MWh)",
    xaxis_title="Trade Date",
    yaxis_title="Price (EUR/MWh)",
    height=480,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.25),
)
fig_norm.add_hline(y=0, line_dash="dot", line_color="black", line_width=1)
fig_norm.update_layout(
    title="Contract Price Evolution -- Normalized (% change from first observation)",
    xaxis_title="Trade Date",
    yaxis_title="% change from first quote",
    height=480,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.25),
)
fig_abs.show()
fig_norm.show()